# 충북대학교 머신러닝 과제 2 — YOLOv8 Fine-tuning
이 노트북은 저장소의 재현 스크립트를 Google Colab CUDA 환경에서 실행하기 위한 보조 진입점입니다. 로컬 최종 제출 결과는 Apple M1 Pro MPS에서 생성되었으므로, Colab 재실행 결과의 마지막 자릿수와 학습 시간은 달라질 수 있습니다.

## 0. 준비
Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤, 공개 저장소를 clone하거나 프로젝트 폴더를 Google Drive에 복사하세요. 강의 PDF는 저장소에 포함하지 않습니다.

In [ ]:
# 공개 후 아래 URL을 실제 GitHub URL로 바꾸세요.
REPO_URL = "https://github.com/USER/REPOSITORY.git"
!git clone $REPO_URL project
%cd project
!pip install -q -r requirements.txt

## 1. 데이터 다운로드, 재매핑 및 검증

In [ ]:
!python src/download_data.py --asset all
!python src/prepare_dataset.py
!python src/validate_dataset.py

In [ ]:
import json
validation = json.load(open('results/metrics/dataset_validation.json'))
assert validation['valid']
validation['splits']

## 2. 학습
`src/train.py`의 제출 실험은 MPS 전용 검증을 포함합니다. Colab CUDA에서는 아래 Ultralytics 명령을 사용하되, 나머지 하이퍼파라미터는 `configs/train.yaml`과 동일하게 유지합니다.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
result = model.train(
    data='data/processed/kitti3/data.yaml', epochs=30, batch=8, imgsz=640,
    optimizer='AdamW', lr0=1e-3, lrf=0.01, weight_decay=5e-4,
    warmup_epochs=3, patience=8, seed=42, deterministic=True,
    device=0, project='results/runs', name='colab_cuda', plots=True,
)

## 3. 독립 test 평가
학습 후 best.pt를 `models/best.pt`에 복사하고 평가합니다. 로컬 제출 결과를 덮어쓰지 않도록 별도 브랜치/폴더 사용을 권장합니다.

In [ ]:
!mkdir -p models
!cp results/runs/colab_cuda/weights/best.pt models/best.pt
from ultralytics import YOLO
best = YOLO('models/best.pt')
metrics = best.val(data='data/processed/kitti3/data.yaml', split='test', device=0, imgsz=640, batch=8, plots=True)
print({'precision': metrics.box.mp, 'recall': metrics.box.mr, 'mAP50': metrics.box.map50, 'mAP50-95': metrics.box.map})